In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error

import lightgbm as lgb

import warnings
warnings.filterwarnings("ignore")

%matplotlib inline

In [ ]:
train_df = pd.read_csv("/content/drive/MyDrive/python-for-biginner/House Prices/train.csv")
train_df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [ ]:
train_df.shape

(1460, 81)

In [ ]:
train_df.dtypes

,0
Id,int64
MSSubClass,int64
MSZoning,object
LotFrontage,float64
LotArea,int64
Street,object
Alley,object
LotShape,object
LandContour,object
Utilities,object


In [ ]:
pd.set_option('display.max_rows', 100)

In [ ]:
train_df.isnull().sum()

,0
Id,0
MSSubClass,0
MSZoning,0
LotFrontage,259
LotArea,0
Street,0
Alley,1369
LotShape,0
LandContour,0
Utilities,0


In [ ]:
train_df["OverallQual"].head()

,OverallQual
0,7
1,6
2,7
3,7
4,8


In [ ]:
train_df["MSSubClass"].head()

,MSSubClass
0,60
1,20
2,60
3,70
4,60


In [ ]:
#特徴量
features = ["OverallQual", "MSSubClass"]

X_train = train_df[features] #全体品質,居住面積
y_train = np.log1p(train_df["SalePrice"]) #対数変換しない？

In [ ]:
print(X_train.shape)
print(y_train.shape)

(1460, 2)
(1460,)


In [ ]:
params = {
    "boosting_type" : "gbdt",
    "objective" : "regression",
    "metric" : "rmse",
    "learning_rate" : 0.1,
    "num_leaves" : 16,
    "n_estimators" : 100000,
    "random_state" : 123,
    "importance_type" : "gain",
}

In [ ]:
#cv
n_splits = 5
cv = KFold(n_splits=n_splits, shuffle=True, random_state=123)

metrics = []
imp = pd.DataFrame()

for nfold, (train_idx, val_idx) in enumerate(cv.split(X_train)):
  x_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
  x_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]

  print("学習用:", x_tr.shape, y_tr.shape)
  print("評価用:", x_va.shape, y_va.shape)

  model = lgb.LGBMRegressor(**params)
  model.fit(
    x_tr,
    y_tr,
    eval_set=[(x_tr, y_tr), (x_va,y_va)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=100, verbose=True),
        lgb.log_evaluation(0),
    ])

  y_tr_pred = model.predict(x_tr)
  y_va_pred = model.predict(x_va)

  rmse_tr = np.sqrt(mean_squared_error(y_tr, y_tr_pred))
  rmse_va = np.sqrt(mean_squared_error(y_va, y_va_pred))

  print("[RMSE] tr: {:.2f}, va: {:.2f}".format(rmse_tr, rmse_va))

  metrics.append([nfold, rmse_tr, rmse_va])

  _imp = pd.DataFrame({
        "col": X_train.columns,
        "imp": model.feature_importances_,
        'nfold': nfold
        })

  imp = pd.concat([imp, _imp], axis=0, ignore_index=True)

学習用: (1168, 2) (1168,)
評価用: (292, 2) (292,)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000455 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 26
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 2
[LightGBM] [Info] Start training from score 12.021806
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[82]	training's rmse: 0.205543	valid_1's rmse: 0.202439
[RMSE] tr: 0.21, va: 0.20
学習用: (1168, 2) (1168,)
評価用: (292, 2) (292,)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000048 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 25
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 2
[LightGBM] [Info] Start training from score 12.014343
Training until validation scores don't improve for 100 

In [ ]:
metrics = np.array(metrics)
print(metrics)

print("[cv ] tr: {:.2f}+-{:.2f}, va: {:.2f}+-{:.2f}".format(
    metrics[:,1].mean(), metrics[:,1].std(),
    metrics[:,2].mean(), metrics[:,2].std(),
))
#CV結果を表にして、平均＋ーばらつきで見る

[[0.         0.20554275 0.20243881]
 [1.         0.20461929 0.21630446]
 [2.         0.2058447  0.21432422]
 [3.         0.20102835 0.22307941]
 [4.         0.21095523 0.20160804]]
[cv ] tr: 0.21+-0.00, va: 0.21+-0.01


In [ ]:
imp = imp.groupby("col")["imp"].agg(["mean", "std"])
imp.columns = ["imp", "imp_std"]

imp_df = imp.sort_values(by='imp', ascending=False)
imp_df.head(30)
#特徴量重要度をnfold平均で安定化して見る

,imp,imp_std
col,,
OverallQual,649.948566,14.607817
MSSubClass,70.142910,5.246156


In [ ]:
test = pd.read_csv('/content/drive/MyDrive/python-for-biginner/House Prices/test.csv')
test.shape

(1459, 80)

In [ ]:
X_test = test[features]
X_test.shape

(1459, 2)

In [ ]:
y_test_pred = model.predict(X_test)
y_test_pred = np.expm1(y_test_pred)

df_submit = pd.DataFrame({
    'Id': test.Id,
    'SalePrice': y_test_pred
})
df_submit

,Id,SalePrice
0,1461,134841.104013
1,1462,166453.434672
2,1463,147577.002743
3,1464,182261.677921
4,1465,226514.557723
...,...,...
1454,2915,96862.705023
1455,2916,96862.705023
1456,2917,134841.104013
1457,2918,141782.679403


In [ ]:
df_submit.to_csv("hp-submission_baseline2.csv", index=False)

In [ ]:
from google.colab import files
files.download("hp-submission_baseline2.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>